In [2]:
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_excel('ua_behavior_merged_may_org.xlsx',sheet_name='ua_behavior_merged_may(1)')  # 或 .csv 用 pd.read_csv

# ===== Step 1: 按user_id聚合 =====
user_agg = df.groupby('user_id').agg(
    total_diamond=('gift_total_diamond_x5', 'sum'),
    play_minutes=('play_min', 'sum'),
    anchor_count=('anchor_id', 'nunique'),  # 打赏主播数
    period_count=('period_order', 'nunique'),  # 活跃切片数
    lucky_diamond=('lucky_send_gift_diamond', 'sum'),
    normal_diamond=('normal_send_gift_diamond', 'sum'),
).reset_index()

# ===== Step 2: 消费集中度 =====
# 每个user在单个主播上的最大消费
max_per_anchor = df.groupby(['user_id', 'anchor_id'])['gift_total_diamond_x5'].sum().reset_index()
max_single_anchor = max_per_anchor.groupby('user_id')['gift_total_diamond_x5'].max().reset_index()
max_single_anchor.columns = ['user_id', 'max_anchor_diamond']

user_agg = user_agg.merge(max_single_anchor, on='user_id', how='left')
user_agg['concentration'] = user_agg['max_anchor_diamond'] / user_agg['total_diamond']

# ===== Step 3: lucky占比 =====
user_agg['lucky_ratio'] = user_agg['lucky_diamond'] / user_agg['total_diamond']

# 每天平均看播时长
user_agg['play_minutes_daily'] = user_agg['play_minutes'] / 10


In [3]:
# 看分布
print(user_agg['play_minutes_daily'].describe())
print(f"日均看播25分位: {user_agg['play_minutes_daily'].quantile(0.25):.2f}")
print(f"日均看播中位数: {user_agg['play_minutes_daily'].quantile(0.5):.2f}")
print(f"日均看播75分位: {user_agg['play_minutes_daily'].quantile(0.75):.2f}")
print(f"日均看播90分位: {user_agg['play_minutes_daily'].quantile(0.90):.2f}")

count    86153.000000
mean        23.690575
std         76.050612
min          0.000000
25%          0.518000
50%          1.889000
75%          9.615000
max       2101.125000
Name: play_minutes_daily, dtype: float64
日均看播25分位: 0.52
日均看播中位数: 1.89
日均看播75分位: 9.62
日均看播90分位: 53.66


In [16]:
# ===== Step 4: 计算门槛 =====
diamond_p30   = user_agg['total_diamond'].quantile(0.30)   # 路人线上移
diamond_p55   = user_agg['total_diamond'].quantile(0.55)   # 低价/高潜分界
diamond_p80   = user_agg['total_diamond'].quantile(0.80)   # 高潜/高价值分界
diamond_p95   = user_agg['total_diamond'].quantile(0.95)   # 顶级消费线
anchor_median  = user_agg['anchor_count'].median()
play_median    = user_agg['play_minutes_daily'].median()   # 日均看播时长中位数
concentration_high = 0.6

print(f"消费30分位(路人线):    {diamond_p30:.0f}")
print(f"消费55分位(低价/高潜): {diamond_p55:.0f}")
print(f"消费80分位(高潜/高价值): {diamond_p80:.0f}")
print(f"消费95分位(顶级消费线): {diamond_p95:.0f}")
print(f"主播数中位数:           {anchor_median:.0f}")
print(f"日均看播时长中位数(分钟): {play_median:.0f}")

# ===== Step 5: 打标签 =====
def label_user(row):
    d = row['total_diamond']
    a = row['anchor_count']
    c = row['concentration']
    p = row['play_minutes_daily']

    # ① 路人：消费极低
    if d < diamond_p30:
        return '路人'

    # ② 正常（大R）：消费极高（p95以上）OR 消费高+集中+时长长
    elif d >= diamond_p95 or (d >= diamond_p80 and c >= concentration_high and p >= play_median):
        return '正常'

    # ③ 高价值分散用户：消费高 + 主播分散 + 时长长（但消费不到顶级）
    elif d >= diamond_p80 and a >= anchor_median and p >= play_median:
        return '高价值分散用户'

    # ④ 高潜韭菜：消费中等偏高 + 分散 + 时长短
    elif d >= diamond_p55 and a >= anchor_median and p < play_median:
        return '高潜韭菜'

    # ⑤ 低价韭菜：其余有消费行为的
    else:
        return '低价韭菜'

user_agg['user_type'] = user_agg.apply(label_user, axis=1)

消费30分位(路人线):    15
消费55分位(低价/高潜): 65
消费80分位(高潜/高价值): 863
消费95分位(顶级消费线): 20166
主播数中位数:           2
日均看播时长中位数(分钟): 2


In [18]:
# ===== Step 6: 汇总统计 =====
summary = user_agg.groupby('user_type').agg(
    人数=('user_id', 'count'),
    总消费=('total_diamond', 'sum'),
    人均消费=('total_diamond', 'mean'),
    消费中位数=('total_diamond', 'median'),
    平均主播数=('anchor_count', 'mean'),
    平均日均看播时长=('play_minutes_daily', 'mean'),
    平均lucky占比=('lucky_ratio', 'mean'),
).reset_index()

summary['消费占比'] = summary['总消费'] / summary['总消费'].sum()
summary['人数占比'] = summary['人数'] / summary['人数'].sum()

summary = summary.sort_values('人均消费', ascending=False).reset_index(drop=True)

display(summary)

# ===== Step 7: 输出结果 =====
# user_agg.to_excel('user_type_labeled.xlsx', index=False)
# summary.to_excel('summary_by_type.xlsx', index=False)

,user_type,人数,总消费,人均消费,消费中位数,平均主播数,平均日均看播时长,平均lucky占比,消费占比,人数占比
0,正常,10514,980906275,93295.25,10517.00,15.17,114.59,0.06,0.96,0.12
1,高价值分散用户,5985,33020574,5517.22,3585.00,22.87,85.64,0.06,0.03,0.07
2,高潜韭菜,3448,1385883,401.94,135.00,3.69,1.06,0.02,0.00,0.04
3,低价韭菜,41057,6780537,165.15,53.00,3.53,7.11,0.03,0.01,0.48
4,路人,25149,162204,6.45,5.00,1.12,1.12,0.01,0.00,0.29
